In [ ]:
import os
home_dir = next((path for path in ["/content", "/kaggle/working"] if os.path.exists(path)), os.getcwd())
os.chdir(home_dir)
!git clone https://github.com/Torfinhell/Diffusion-Trajectory-Forecaster.git
!cd Diffusion-Trajectory-Forecaster

In [ ]:
!pip install uv
!uv sync

## Setup & Workflow

#### Docker

In [ ]:
#recommended only for server, wont work in kaggle or colab
!docker build -t diffusion-trajectory-forecaster .
!chmod +x scripts/docker_run.sh
!scripts/docker_run.sh bash

#### Authorization

In [ ]:
import os

def set_credentials(aws_access_key_id=None, aws_secret_access_key=None, gh_token=None, git_username=None, git_email=None, gcp_credentials_json=None):
    try:
        from google.colab import userdata, auth
        os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
        os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
        os.environ["GITHUB_TOKEN"] = userdata.get('GH_TOKEN')
        os.environ["GIT_USERNAME"] = userdata.get('GIT_USERNAME')
        os.environ["GIT_EMAIL"] = userdata.get('GIT_EMAIL')
        print("✅ Google Colab detected. Secrets loaded.")
        auth.authenticate_user()
        print("✅ Google Cloud User Account authenticated.")
    except ImportError:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            os.environ["AWS_ACCESS_KEY_ID"] = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
            os.environ["AWS_SECRET_ACCESS_KEY"] = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
            os.environ["GITHUB_TOKEN"] = user_secrets.get_secret("GH_TOKEN")
            os.environ["GIT_USERNAME"] = user_secrets.get_secret("GIT_USERNAME")
            os.environ["GIT_EMAIL"] = user_secrets.get_secret("GIT_EMAIL")
            gcp_json_str = user_secrets.get_secret("GCP_CREDENTIALS_JSON")
            print("✅ Kaggle detected. Secrets loaded.")
            
            cred_path = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
            os.makedirs(os.path.dirname(cred_path), exist_ok=True)
            with open(cred_path, "w") as f:
                f.write(gcp_json_str)
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cred_path
            os.system(f"gcloud auth activate-service-account --key-file={cred_path} --quiet")
            print("✅ Google Cloud Service Account authenticated via ADC.")
        except Exception:
            assert aws_access_key_id is not None, "aws_access_key_id cannot be None in local environment"
            assert aws_secret_access_key is not None, "aws_secret_access_key cannot be None in local environment"
            assert gh_token is not None, "gh_token cannot be None in local environment"
            assert git_username is not None, "git_username cannot be None in local environment"
            assert git_email is not None, "git_email cannot be None in local environment"
            assert gcp_credentials_json is not None, "gcp_credentials_json cannot be None in local environment"
            
            os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key_id
            os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_access_key
            os.environ["GITHUB_TOKEN"] = gh_token
            os.environ["GIT_USERNAME"] = git_username
            os.environ["GIT_EMAIL"] = git_email
            print("✅ Local environment detected. Manual keys loaded.")
            
            cred_path = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
            os.makedirs(os.path.dirname(cred_path), exist_ok=True)
            with open(cred_path, "w") as f:
                f.write(gcp_credentials_json)
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cred_path
            os.system(f"gcloud auth activate-service-account --key-file={cred_path} --quiet")
            print("✅ Google Cloud Service Account authenticated locally via ADC.")

    os.system(f'git config --global user.name "{os.environ.get("GIT_USERNAME")}"')
    os.system(f'git config --global user.email "{os.environ.get("GIT_EMAIL")}"')
    os.system('git config --global credential.helper store')
    print("✅ Git identity and configuration applied.")

set_credentials()


## Creating Dataset

In [ ]:
!uv run python -m scripts.create_dataset dataset=small_no_scenes dataset/feat_extract=default
!uv run scripts/add_local_dataset_to_dvc.sh data/small_no_scenes
# !git commit -m "Added new dvc"
# !git push

## Training

## Profiling

## Quantization

## Distillation

## Inferencing and visualization

#### Download Checkpoints

#### Inference

## Running the application